# Contextual Toxic Comment Classification Model

This notebook evaluates whether a contextual language model can improve harmful-content detection compared with the TF-IDF and Logistic Regression baseline.

Error analysis of the baseline moderation system showed that the largest remaining weakness was detecting toxic-only comments. Many missed toxic comments received predicted probabilities substantially below the selected classification threshold, suggesting that threshold adjustment alone may not resolve the remaining errors.

A contextual language model will therefore be evaluated to determine whether representing the meaning and context of language improves harmful-content detection, particularly for comments missed by the baseline model.

In [ ]:
!nvidia-smi

Wed Oct 28 23:26:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


In [ ]:
!pip install -q -U transformers datasets evaluate accelerate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [ ]:
import torch
import transformers
import datasets
import evaluate
import accelerate
import pyarrow

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Evaluate:", evaluate.__version__)
print("Accelerate:", accelerate.__version__)
print("PyArrow:", pyarrow.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch: 2.11.0+cu130
Transformers: 5.21.0
Datasets: 5.1.0
Evaluate: 0.4.6
Accelerate: 1.15.0
PyArrow: 26.0.0
GPU available: True


In [ ]:
import pandas as pd

df = pd.read_csv("/content/train.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (159571, 8)


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [ ]:
from sklearn.model_selection import train_test_split

label_columns = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

X = df["comment_text"]
y = df[label_columns]

is_harmful = (y.sum(axis=1) > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=is_harmful
)

In [ ]:
train_is_harmful = (y_train.sum(axis=1) > 0).astype(int)

X_model_train, X_validation, y_model_train, y_validation = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=train_is_harmful
)

print("Model training:", X_model_train.shape)
print("Validation:", X_validation.shape)
print("Final test:", X_test.shape)

Model training: (102124,)
Validation: (25532,)
Final test: (31915,)


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

print("Tokenizer loaded.")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded.


In [ ]:
example_comment = X_model_train.iloc[0]

print("Original comment:")
print(example_comment)

print("\nTokens:")
print(tokenizer.tokenize(example_comment))

print("\nToken IDs:")
print(tokenizer.encode(example_comment))

Original comment:
"

 Regarding your edits to Ellen Pompeo and Patrick Dempsey 

I have reverted your edits to these two articles.  Please review WP:V to learn about appropriate, reliable sources for the information you keep adding, and also visit WP:LIBEL and WP:BLP to learn about appropriate information in biographical articles of living people.  Insinuating that these two individuals are having an affair or are anything other than faithful to their respective significant others is libelous, and against Wikipedia policy.  You have not provided a reliable source to support the POV suggested by your edits.  WP:NPOV will help you learn about Wikipedia's neutral point of view policy.  

The quotes you keep adding are taken out of contexttheir mentions of a ""crush"" on each other were done in a joking manner, and another cast member followed their comments by saying she had a crush on both of thembut that's not grounds for speculating that the cast member in question is having an affair 

In [ ]:
example_tokens = tokenizer.encode(example_comment)

print("Number of tokens:", len(example_tokens))

Number of tokens: 393


In [ ]:
token_lengths = []

for comment in X_model_train:
    tokens = tokenizer.encode(
        comment,
        add_special_tokens=True,
        truncation=False
    )
    token_lengths.append(len(tokens))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (849 > 512). Running this sequence through the model will result in indexing errors


In [ ]:
pd.Series(token_lengths).describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

,0
count,102124.000000
mean,94.852963
std,141.401275
min,4.000000
50%,52.000000
75%,104.000000
90%,206.000000
95%,310.000000
99%,801.000000
max,4950.000000


In [ ]:
length_analysis = pd.DataFrame({
    "token_length": token_lengths,
    "is_harmful": (
        y_model_train[label_columns].sum(axis=1) > 0
    ).astype(int).values
})

length_analysis.groupby("is_harmful")["token_length"].describe()

,count,mean,std,min,25%,50%,75%,max
is_harmful,,,,,,,,
0,91740.0,96.326608,135.250765,4.0,27.0,54.0,107.0,3394.0
1,10384.0,81.833686,186.659036,4.0,19.0,36.0,73.0,4950.0


In [ ]:
for max_length in [128, 256, 512]:
    overall = (length_analysis["token_length"] > max_length).mean() * 100

    harmful = (
        length_analysis.loc[
            length_analysis["is_harmful"] == 1,
            "token_length"
        ] > max_length
    ).mean() * 100

    non_harmful = (
        length_analysis.loc[
            length_analysis["is_harmful"] == 0,
            "token_length"
        ] > max_length
    ).mean() * 100

    print(f"Max length {max_length}")
    print(f"  Overall truncated:     {overall:.2f}%")
    print(f"  Harmful truncated:     {harmful:.2f}%")
    print(f"  Non-harmful truncated: {non_harmful:.2f}%")
    print()

Max length 128
  Overall truncated:     19.36%
  Harmful truncated:     12.74%
  Non-harmful truncated: 20.11%

Max length 256
  Overall truncated:     6.92%
  Harmful truncated:     4.84%
  Non-harmful truncated: 7.15%

Max length 512
  Overall truncated:     2.20%
  Harmful truncated:     2.19%
  Non-harmful truncated: 2.20%



### Maximum Sequence Length

The token-length distribution was examined before selecting the maximum sequence length for DistilBERT.

Harmful comments were generally shorter than non-harmful comments. Harmful comments had a median length of 36 tokens compared with 54 tokens for non-harmful comments.

Three possible maximum sequence lengths were evaluated. A limit of 128 tokens would truncate 12.74% of harmful comments, while 256 tokens would truncate 4.84%. Increasing the limit to 512 tokens would reduce harmful-comment truncation further to 2.19%, but would increase computational and memory requirements.

A maximum sequence length of **256 tokens** was selected as a balance between preserving comment information and maintaining reasonable computational requirements for model training.

In [ ]:
train_df = pd.DataFrame({
    "comment_text": X_model_train
})

validation_df = pd.DataFrame({
    "comment_text": X_validation
})

train_df["labels"] = y_model_train[label_columns].astype(float).values.tolist()
validation_df["labels"] = y_validation[label_columns].astype(float).values.tolist()

train_df.head()

,comment_text,labels
30216,"""\n\n Regarding your edits to Ellen Pompeo and...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
112011,"""\n\n Do-support and the three-revert rule \n\...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
59819,That I consider as a serious offence. What - w...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
3028,"delete the biased reference or block me,there'...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
36598,"""\n\n """"ROTFL""""???? \n\nthe article currently ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"


In [ ]:
print("Comment:")
print(train_df.iloc[0]["comment_text"])

print("\nLabels:")
print(train_df.iloc[0]["labels"])

print("\nLabel order:")
print(label_columns)

Comment:
"

 Regarding your edits to Ellen Pompeo and Patrick Dempsey 

I have reverted your edits to these two articles.  Please review WP:V to learn about appropriate, reliable sources for the information you keep adding, and also visit WP:LIBEL and WP:BLP to learn about appropriate information in biographical articles of living people.  Insinuating that these two individuals are having an affair or are anything other than faithful to their respective significant others is libelous, and against Wikipedia policy.  You have not provided a reliable source to support the POV suggested by your edits.  WP:NPOV will help you learn about Wikipedia's neutral point of view policy.  

The quotes you keep adding are taken out of contexttheir mentions of a ""crush"" on each other were done in a joking manner, and another cast member followed their comments by saying she had a crush on both of thembut that's not grounds for speculating that the cast member in question is having an affair with both

In [ ]:
harmful_example = train_df[
    train_df["labels"].apply(lambda x: sum(x) > 0)
].iloc[0]

print("Comment:")
print(harmful_example["comment_text"])

print("\nLabels:")
print(harmful_example["labels"])

print("\nLabel order:")
print(label_columns)

Comment:
, I hope your retarded kids get anal raped and murdered for having such a fag as a father. I'm gonna fuck your fat wife and trow her over the bridge. Consider yourself happy, another useful editor of Wikiepia retired. I see more users retired everyday. Don't even dare removing this you peace of shit, I do what ever the fuck I want with my talk page. Report all you want old cunt, I no longer give a fuck. And don't fucking leave a NPA warning template, just block me. That's what you do best.

Labels:
[1.0, 1.0, 1.0, 1.0, 1.0, 1.0]

Label order:
['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']


In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

validation_dataset = Dataset.from_pandas(
    validation_df,
    preserve_index=False
)

print(train_dataset)
print(validation_dataset)

Dataset({
    features: ['comment_text', 'labels'],
    num_rows: 102124
})
Dataset({
    features: ['comment_text', 'labels'],
    num_rows: 25532
})


In [ ]:
MAX_LENGTH = 256

def tokenize_batch(batch):
    return tokenizer(
        batch["comment_text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

In [ ]:
tokenized_train = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["comment_text"]
)

tokenized_validation = validation_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["comment_text"]
)

Map:   0%|          | 0/102124 [00:00<?, ? examples/s]

Map:   0%|          | 0/25532 [00:00<?, ? examples/s]

In [ ]:
print(tokenized_train)
print()
print(tokenized_train[0])

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 102124
})

{'labels': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 'input_ids': [101, 1000, 4953, 2115, 10086, 2015, 2000, 9155, 13433, 8737, 8780, 1998, 4754, 28165, 1045, 2031, 16407, 2115, 10086, 2015, 2000, 2122, 2048, 4790, 1012, 3531, 3319, 1059, 2361, 1024, 1058, 2000, 4553, 2055, 6413, 1010, 10539, 4216, 2005, 1996, 2592, 2017, 2562, 5815, 1010, 1998, 2036, 3942, 1059, 2361, 1024, 26201, 1998, 1059, 2361, 1024, 1038, 14277, 2000, 4553, 2055, 6413, 2592, 1999, 16747, 4790, 1997, 2542, 2111, 1012, 16021, 2378, 24133, 2008, 2122, 2048, 3633, 2024, 2383, 2019, 6771, 2030, 2024, 2505, 2060, 2084, 11633, 2000, 2037, 7972, 3278, 2500, 2003, 26201, 3560, 1010, 1998, 2114, 16948, 3343, 1012, 2017, 2031, 2025, 3024, 1037, 10539, 3120, 2000, 2490, 1996, 13433, 2615, 4081, 2011, 2115, 10086, 2015, 1012, 1059, 2361, 1024, 27937, 4492, 2097, 2393, 2017, 4553, 2055, 16948, 1005, 1055, 8699, 2391, 1997, 3193,

In [ ]:
tokenized_train = tokenized_train.remove_columns(
    ["token_type_ids"]
)

tokenized_validation = tokenized_validation.remove_columns(
    ["token_type_ids"]
)

print(tokenized_train)

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 102124
})


In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=6,
    problem_type="multi_label_classification"
)

print("Problem type:", model.config.problem_type)
print("Number of labels:", model.config.num_labels)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Problem type: multi_label_classification
Number of labels: 6


In [ ]:
id2label = {
    0: "toxic",
    1: "severe_toxic",
    2: "obscene",
    3: "threat",
    4: "insult",
    5: "identity_hate"
}

label2id = {
    label: index
    for index, label in id2label.items()
}

model.config.id2label = id2label
model.config.label2id = label2id

print(model.config.id2label)

{0: 'toxic', 1: 'severe_toxic', 2: 'obscene', 3: 'threat', 4: 'insult', 5: 'identity_hate'}


In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("Data collator created.")

Data collator created.


In [ ]:
sample_batch = data_collator([
    tokenized_train[1],
    tokenized_train[2],
    tokenized_train[3]
])

print("Batch input shape:", sample_batch["input_ids"].shape)
print("Batch label shape:", sample_batch["labels"].shape)

Batch input shape: torch.Size([3, 158])
Batch label shape: torch.Size([3, 6])


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./distilbert_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    logging_steps=500,
    save_total_limit=1,
    report_to="none"
)

print("Training configuration created.")

Training configuration created.


In [ ]:
print("Label data type:", sample_batch["labels"].dtype)

Label data type: torch.float32


In [ ]:
import numpy as np

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probabilities = 1 / (1 + np.exp(-logits))
    predictions = (probabilities >= 0.50).astype(int)

    macro_precision = precision_score(
        labels, predictions, average="macro", zero_division=0
    )

    macro_recall = recall_score(
        labels, predictions, average="macro", zero_division=0
    )

    macro_f1 = f1_score(
        labels, predictions, average="macro", zero_division=0
    )

    actual_harmful = (labels.sum(axis=1) > 0).astype(int)
    predicted_harmful = (predictions.sum(axis=1) > 0).astype(int)

    moderation_precision = precision_score(
        actual_harmful, predicted_harmful, zero_division=0
    )

    moderation_recall = recall_score(
        actual_harmful, predicted_harmful, zero_division=0
    )

    moderation_f1 = f1_score(
        actual_harmful, predicted_harmful, zero_division=0
    )

    return {
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "moderation_precision": moderation_precision,
        "moderation_recall": moderation_recall,
        "moderation_f1": moderation_f1
    }

print("Evaluation function created.")

Evaluation function created.


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("Trainer created.")
print("Model device:", next(trainer.model.parameters()).device)

Trainer created.
Model device: cuda:0


In [ ]:
import torch

test_batch = data_collator([
    tokenized_train[0],
    tokenized_train[1],
    tokenized_train[2],
    tokenized_train[3]
])

test_batch = {
    key: value.to(model.device)
    for key, value in test_batch.items()
}

model.eval()

with torch.no_grad():
    outputs = model(**test_batch)

print("Loss:", outputs.loss.item())
print("Logits shape:", outputs.logits.shape)

Loss: 0.72064208984375
Logits shape: torch.Size([4, 6])


In [ ]:
training_result = trainer.train()

Epoch,Training Loss,Validation Loss,Macro Precision,Macro Recall,Macro F1,Moderation Precision,Moderation Recall,Moderation F1
1,0.039772,0.038272,0.732558,0.492004,0.553793,0.924330,0.743451,0.824082
2,0.030793,0.037942,0.719558,0.600088,0.640142,0.869300,0.822419,0.845210


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
prediction_output = trainer.predict(tokenized_validation)

validation_logits = prediction_output.predictions

validation_probabilities = 1 / (
    1 + np.exp(-validation_logits)
)

validation_predictions = (
    validation_probabilities >= 0.50
).astype(int)

print("Probability shape:", validation_probabilities.shape)
print("Prediction shape:", validation_predictions.shape)

Probability shape: (25532, 6)
Prediction shape: (25532, 6)


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os

save_path = "/content/drive/MyDrive/social-media-comment-moderator/distilbert_model"

os.makedirs(save_path, exist_ok=True)

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Model saved to:", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/drive/MyDrive/social-media-comment-moderator/distilbert_model


In [ ]:
import numpy as np

np.save(
    "/content/drive/MyDrive/social-media-comment-moderator/validation_probabilities.npy",
    validation_probabilities
)

print("Validation probabilities saved.")

Validation probabilities saved.
